# Pipeline 验证

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import SimpleITK as sitk
import skimage

try:
    import tigre
    tigre_version = getattr(tigre, "__version__", "unknown")
except Exception as exc:
    tigre_version = f"unavailable: {exc}"

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("SimpleITK:", sitk.Version())
print("skimage:", skimage.__version__)
print("tigre:", tigre_version)


## 1. 加载预处理后的 CT

In [ ]:
processed_root = Path("data/processed")
case_dirs = sorted([p for p in processed_root.iterdir() if p.is_dir()])
if not case_dirs:
    raise FileNotFoundError("No processed case found under data/processed")

case_dir = case_dirs[0]
volume = np.load(case_dir / "volume.npy")
seg_path = case_dir / "seg.npy"
seg = np.load(seg_path) if seg_path.exists() else None

print("case:", case_dir.name)
print("volume shape:", volume.shape, "dtype:", volume.dtype, "min/max:", float(volume.min()), float(volume.max()))
if seg is not None:
    print("seg shape:", seg.shape, "dtype:", seg.dtype, "labels:", np.unique(seg)[:10])
else:
    print("seg.npy not found for this case")


## 2. 可视化 CT 切片

In [ ]:
z, y, x = [s // 2 for s in volume.shape]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(volume[z], cmap="gray")
axes[0].set_title("Axial")
axes[1].imshow(volume[:, y, :], cmap="gray")
axes[1].set_title("Coronal")
axes[2].imshow(volume[:, :, x], cmap="gray")
axes[2].set_title("Sagittal")

if seg is not None:
    axes[0].imshow(seg[z], cmap="jet", alpha=0.25)
    axes[1].imshow(seg[:, y, :], cmap="jet", alpha=0.25)
    axes[2].imshow(seg[:, :, x], cmap="jet", alpha=0.25)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. 投影仿真

In [ ]:
proj_root = Path("data/projections") / case_dir.name
view_dirs = sorted([p for p in proj_root.iterdir() if p.is_dir() and p.name.endswith("views")])
if not view_dirs:
    raise FileNotFoundError(f"No projection results found under {proj_root}")

view_dir = view_dirs[0]
projections = np.load(view_dir / "projections.npy")
angles = np.load(view_dir / "angles.npy")
print("using:", view_dir.name, "projections shape:", projections.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
idxs = [0, len(angles) // 2, len(angles) - 1]
for ax, idx in zip(axes, idxs):
    ax.imshow(projections[idx], cmap="gray")
    ax.set_title(f"angle={np.rad2deg(angles[idx]):.1f}°")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. FDK 重建 vs GT

In [ ]:
import json
from anatcoder.eval.global_metrics import evaluate_reconstruction

fdk = np.load(view_dir / "fdk_recon.npy")
metrics = evaluate_reconstruction(fdk, volume, data_range=1.0)
print(metrics)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(volume[volume.shape[0] // 2], cmap="gray")
axes[0].set_title("GT")
axes[1].imshow(fdk[fdk.shape[0] // 2], cmap="gray")
axes[1].set_title(f"FDK | PSNR={metrics['psnr']:.2f}, SSIM={metrics['ssim']:.3f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Atlas 验证

In [ ]:
atlas_path = Path("data/atlas/oracle_atlas.npy")
if not atlas_path.exists():
    raise FileNotFoundError(f"Atlas not found: {atlas_path}")

atlas = np.load(atlas_path)
print("atlas shape:", atlas.shape, "dtype:", atlas.dtype)

classes_to_show = [1, 2, 3] if atlas.shape[0] > 3 else list(range(atlas.shape[0]))
z = atlas.shape[1] // 2
fig, axes = plt.subplots(1, len(classes_to_show), figsize=(5 * len(classes_to_show), 4))
if len(classes_to_show) == 1:
    axes = [axes]
for ax, cls in zip(axes, classes_to_show):
    ax.imshow(atlas[cls, z], cmap="viridis")
    ax.set_title(f"class {cls} prob")
    ax.axis("off")
plt.tight_layout()
plt.show()
